---
authors:
  - edesz
date: 2025-10-04
---

# Inference

## About

In this step, the best end-to-end ML pipeline that was trained on all available data at the end of the evaluation phase is used to make inference predictions.

Inference is usually performed on data not used during ML model development. However, [as discussed in the project scope](../references/scope/06_analysis.md#model-inference), for this use-case, the final business metrics will be estimated on the all *existing* customer data used to develop the best end-to-end pipeline. For this reason, the inference predictions made in this step will be over the same credit card customer data used to train the best ML pipeline.

:::{note}
### Outputs

Artifacts of the inference Metaflow flow run are stored localy in the `notebooks/.metaflow/InferenceFlow/` directory.

Based on the project deliverables in [this project's scoping document](../references/scope/08_deliverables.md), this step produces following outputs in the R2 bucket

1. The best trained ML model is used to make predictions on the
   - all available data and these predictions and probabilities are exported to a `.parquet.gzip` file. These predictions will be used in the next step to calculate business metrics on all existing customers who have churned.
2. The best trained [`scikit-learn` ML pipeline object is exported](https://scikit-learn.org/stable/model_persistence.html#model-persistence) using the [Python `joblib` library](https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html) after training it on
   - all available data
:::

## Python Imports

Below are the Python package imports required for making inference predictions with Metaflow

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from great_tables import GT, loc, md, style
from metaflow import Flow, Run

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

## User Inputs

Below we define variables that will be used to run the Metaflow inference flow

In [ ]:
# R2 data bucket details
prefix = "cloud-run"
r2_key_train = f"{prefix}/train_data.parquet.gzip"
r2_key_val = f"{prefix}/validation_data.parquet.gzip"
r2_key_test = f"{prefix}/test_data.parquet.gzip"

# datatypes for categorical and ordinal columns
dtypes_ordinals = {
    "income_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}
dtypes_categoricals = {
    "gender": "string[pyarrow]",
    "marital_status": "string[pyarrow]",
    "card_category": "string[pyarrow]",
}

```{attention}
Every parameter from the above cell is also a parameter of the Metaflow inference flow.
```

Below, we extract the other parameters of the Metaflow inference flow

1. [best experiment run ID during validation](./05_get_best_validation_experiment.ipynb#best-validation-outputs)
2. `best_experiment_run_id` during evaluation

from the last run of the Metaflow evaluation flow

In [ ]:
df_eval_run_latest = pd.DataFrame.from_records(
    [
        {
            "run_id": int(run.id),
            "model_name": run.data.best_model_name,
        }
        for k, run in enumerate(list(Flow("EvaluationFlow").runs()), 1)
    ]
).nsmallest(1, "run_id")

We'll now get the best experiment ID (from evaluation) and model name (from validation)

In [ ]:
eval_run_id = df_eval_run_latest["run_id"].astype(str).squeeze()
best_model_name = df_eval_run_latest["model_name"].squeeze()

## Inference

### Flow Definition

The Metaflow flow for inference contains the following steps

1. `start`
   - extract best end-to-end pipeline and decision threshold and Metaflow run object determined in the validation phase, from the Metaflow evaluation flow
2. `extract`
   - inference data from R2 bucket and separate features (`X_test`) from class labels (`y_test`)
   - for the use-case here, this is all available data - namely, the same data used to train the best end-to-end pipeline in preparation for inference
3. `predict_proba`
   - predict probabilities for all available data
4. `predict`
   - convert predicted probabilities into hard labels
5. `gather`
   - combine data and predicted probabilities
6. `export`
   - exports project deliverables to private R2 bucket
   - predictions with the following columns
     - `y_pred`
     - `y_pred_proba`
     - `model_name`
     - `best_decision_threshold`
   - pipeline trained on all available data

(inference-flow)=
The inference flow is defined below

In [ ]:
import json
import os
from datetime import datetime

import boto3
import pandas as pd
import r2.io_utils as r2io
from cc_churn.mflow_utils import get_metaflow_run_artifacts
from metaflow import FlowSpec, NBRunner, Parameter, Run, step


class InferenceFlow(FlowSpec):
    r2_keys = Parameter(
        name="r2_keys",
        help="R2 keys",
        default="{'train': 'C', 'val': 'D', 'test': 'E'}",
    )
    dtypes_ordinals = Parameter(
        name="dtypes_ordinals",
        help="ordinal datatypes",
        default="{'B': 'string[pyarrow]'}",
    )
    dtypes_categoricals = Parameter(
        name="dtypes_categoricals",
        help="categorical datatypes",
        default="{'G': 'string[pyarrow]'}",
    )
    best_model_name = Parameter(
        name="best_model_name",
        help="best model name from validation",
        default="LogisticRegression",
    )
    eval_run_id = Parameter(
        name="eval_run_id", help="Metaflow Run ID", default="138473443"
    )

    @step
    def start(self):
        run_eval = Run(f"EvaluationFlow/{self.eval_run_id}")
        self.threshold = run_eval.data.threshold
        self.pipe = run_eval.data.pipe
        self.next(self.extract)

    @step
    def extract(self):
        self.boto3_params = dict(
            service_name="s3",
            endpoint_url=(
                f"https://{os.getenv('ACCOUNT_ID')}.r2.cloudflarestorage.com"
            ),
            aws_access_key_id=os.getenv("ACCESS_KEY_ID_USER2"),
            aws_secret_access_key=os.getenv("SECRET_ACCESS_KEY_USER2"),
            region_name="auto",
        )
        df = pd.concat(
            [
                (
                    r2io.pandas_read_parquet_r2(
                        s3_client=boto3.client(**self.boto3_params),
                        bucket_name=os.getenv("BUCKET_NAME"),
                        r2_key=json.loads(self.r2_keys)[key],
                    )
                    .astype(json.loads(self.dtypes_ordinals))
                    .astype(json.loads(self.dtypes_categoricals))
                )
                for key in ["train", "val", "test"]
            ]
        )

        self.X = df.drop(columns=["is_churned"])
        self.y = df["is_churned"]
        self.next(self.pred_proba)

    @step
    def pred_proba(self):
        self.y_pred_proba = pd.Series(
            self.pipe.predict_proba(self.X)[:, 1],
            index=self.X.index,
            dtype="float64[pyarrow]",
        )
        self.next(self.predict)

    @step
    def predict(self):
        self.y_pred = self.y_pred_proba >= self.threshold
        self.next(self.gather)

    @step
    def gather(self):
        dtypes_categoricals = {
            "marital_status": "category",
            "card_category": "category",
        }
        dtype_predictions_metadata = {
            "model_name": "category",
            "best_decision_threshold": "float64[pyarrow]",
        }
        self.df_true_pred = (
            pd.concat(
                [
                    self.X.assign(
                        y_pred=self.y_pred, y_pred_proba=self.y_pred_proba
                    ),
                    self.y,
                ],
                axis=1,
            )
            .assign(
                model_name=self.best_model_name,
                best_decision_threshold=self.threshold,
            )
            .astype(dtype_predictions_metadata)
            .astype(dtypes_categoricals)
        )
        self.next(self.export)

    @step
    def export(self):
        prefix = "cloud-run"
        curr_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        folder = f"{prefix}/{datetime.now().strftime('%Y-%m-%d')}"

        r2io.export_df_to_r2(
            s3_client=boto3.client(**self.boto3_params),
            df=self.df_true_pred,
            bucket_name=os.getenv("BUCKET_NAME"),
            r2_key=(
                f"{folder}/all_predictions__{self.best_model_name.lower()}__"
                f"{curr_timestamp}.parquet.gzip"
            ),
            verbose=False,
        )

        r2io.joblib_dump_to_r2(
            s3_client=boto3.client(**self.boto3_params),
            pipe=self.pipe,
            bucket_name=os.getenv("BUCKET_NAME"),
            r2_key=(
                f"{folder}/best_model__{self.best_model_name}__all__"
                f"{curr_timestamp}.joblib"
            ),
            verbose=False,
        )
        self.next(self.end)

    @step
    def end(self):
        pass


_ = NBRunner(InferenceFlow, pylint=False).nbrun(
    r2_keys=json.dumps(
        {"train": r2_key_train, "val": r2_key_val, "test": r2_key_test}
    ),
    dtypes_ordinals=json.dumps(dtypes_ordinals),
    dtypes_categoricals=json.dumps(dtypes_categoricals),
    best_model_name=best_model_name,
    eval_run_id=eval_run_id,
)

```{note}
In the `extract()` step of this flow, all three processed data splits (train, validation and test) are loaded and combined.

The `predict_proba()` and `predict()` steps make soft and hard predictions respectively.

In the `gather()` step, the hard and soft predictions are appended to the features for all customers in an instance variable `DataFrame` named `df_true_pred`. Inference flow run metadata (`feat_group` and `model_name`) is also appended to `df_true_pred`. Since it is an instance variable, it will be automatically persisted to disk.

The `export` step saves the inference predictions and pipeline used, to the R2 bucket.
```

```{hint} Single Run of Inference Flow
In this step, the Metaflow evaluation flow only runs once, with a fixed set of parameters, to make inference predictions. If the `r2_keys` parameter is changed, it can run with different input data.
```

### Sanity Checks for Flow Execution

We'll now ensure the inference flow ran as expected.

First, we'll extract the run ID and run object from the Metaflow inference flow run, and verify the run completed successfully

In [ ]:
# get all Metaflow inference flow runs
runs_infer = list(Flow("InferenceFlow").runs())

# get ID of Metaflow inference flow run
run_infer_id = runs_infer[0].id

# get Metaflow inference flow run
run_infer = Run(f"InferenceFlow/{run_infer_id}")

# verify run completed
assert run_infer.finished == True
assert run_infer.successful == True

Next, we'll get all runs of the Metaflow inference flow

In [ ]:
%%time
df_infer_flow_runs = pd.DataFrame.from_records(
    [
        {
            "id": run.id,
            "started_at": run.created_at,
            "finished_at": run.finished_at,
            'tags': list(run.tags),
            'pathspec': run.pathspec,
            "finished": run.finished,
            "was_successful": run.successful,
        }
        for run in list(Flow("InferenceFlow").runs())
    ]
)

These are shown below

In [ ]:
gt = (
    GT(df_infer_flow_runs)
    .tab_header(md("**Inference Flow Runs**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["id"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["finished", "was_successful"]),
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

Get the most recent Metaflow inference flow run

In [ ]:
%%time
df_infer_run = df_infer_flow_runs.sort_values(
    by=["finished_at"], ascending=False, ignore_index=True
).head(1)

This run is shown below

In [ ]:
gt = (
    GT(df_infer_run)
    .tab_header(md("**Most Recent Run of Inference Flow**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["id"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["finished", "was_successful"]),
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

Verify it was completed successfully

In [ ]:
assert not df_infer_run.empty
assert df_infer_run["finished"].squeeze() == True
assert df_infer_run["was_successful"].squeeze() == True

## Conclusion

The best end-to-end pipeline, as determined using validation and scored during model evaluation and trained on all available data, was used to make inference predictions here. As per the requirements of the business use-case discussed in the project scope, the predictions were made on all available data. Predicted probabilities and predicted class labels were appended to all available data and then exported to the R2 bucket.